# Arm C — Data-Driven Clusters (k-Prototypes)

This notebook implements Arm C of the study: patients are partitioned into data-driven clusters using k-prototypes, and a classifier is trained independently within each cluster. Arm C addresses RQ2 by comparing cluster-stratified modelling against the non-stratified baseline established in Arm A (global_model_armA.ipynb), and RQ3 by comparing it against the sex-stratified subgroups of Arm B (clinical_stratified_armB.ipynb).

k-prototypes is used because the feature set contains both continuous and categorical clinical variables; it combines Euclidean distance on continuous features with simple-matching distance on categorical features, unlike k-means (continuous only) or k-modes (categorical only) (proposal, Section 3.3).

Arm C reuses Arm A's dataset, feature definitions, classifiers, hyperparameter grids, nested cross-validation procedure, evaluation metrics, and outer fold partitions unchanged. The only methodological difference from Arm A is that patients are first grouped into clusters — fitted independently within each outer training fold, never on validation data — and a separate logistic regression and random forest are fitted within each cluster. Predictions from all cluster models are pooled back into a single population-level validation set for the primary comparison against Arm A and Arm B, as required by the proposal (Section 3.5).

**Pre-registered prediction (2026-09-09, written before running this notebook).** Arm A outperformed Arm B by roughly 5.8 accuracy points for logistic regression (0.842 vs 0.784), a gap that traced almost entirely to Arm B halving the training data available to each subgroup model rather than to any absent sex-based interaction signal: the female subgroup's F1 was unstable (0.461 ± 0.428 across folds, including one fold at 0.000), which is the signature of estimation variance from a small, imbalanced training split, not evidence of a real clinical distinction being captured. Arm C's clusters will be smaller than Arm B's 96-patient female subgroup — on 297 patients, k ∈ {2, 3, 4} gives clusters averaging 148, 99, or 74 patients before any imbalance, and k-prototypes has no reason to split evenly — and clusters have no equivalent to sex's ~30-point base-rate separation (55.7% vs 26.0% disease prevalence) to justify the split. **Prediction: Arm C will underperform both Arm A and Arm B on pooled accuracy, F1, and ROC-AUC**, for the same sample-size reason that hurt Arm B, plausibly compounded further. If this prediction holds, it is evidence that the mechanism identified in Arm B — variance cost of stratification exceeding its bias benefit at n≈300 — generalises across stratification strategies rather than being specific to sex. If it does not hold, that is itself a result worth investigating rather than a failure of the prediction; see Section 15.

## 2. Imports and configuration

`RANDOM_STATE`, `K_OUTER`, and `K_INNER` match Arm A and Arm B exactly, for the same reasons documented there. Three additional settings are specific to Arm C's clustering step:

- `K_CLUSTER_CANDIDATES = [2, 3, 4]`: the candidate cluster counts considered within each outer training fold, matching the proposal's requirement to keep the number of clusters small given the dataset size (Section 3.3).
- `MIN_CLUSTER_SIZE = 40`: the minimum number of training patients a cluster must contain before a classifier is fitted within it. This is smaller than Arm B's smallest subgroup (~80 female training patients per fold) because Arm C's clusters are expected to be smaller than Arm B's subgroups; the guard in Section 10 also requires enough patients of each class for 5-fold inner cross-validation to be well-defined, which is a separate and stricter condition than the raw size threshold.
- `N_INIT_KPROTO = 10`: the number of random centroid initialisations k-prototypes runs per candidate k, matching the library default so results are typical rather than best-of-many.

In [1]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, silhouette_score

from kmodes.kprototypes import KPrototypes

# Configuration (RANDOM_STATE, K_OUTER, K_INNER must match Arm A and Arm B)
RANDOM_STATE = 42
K_OUTER      = 5
K_INNER      = 5

# Arm C-specific clustering configuration
K_CLUSTER_CANDIDATES = [2, 3, 4]
MIN_CLUSTER_SIZE      = 40
N_INIT_KPROTO         = 10

## 3. Load dataset

Arm C uses the same cleaned Cleveland extract as Arm A and Arm B. The cleaning procedure is documented in `data_cleaning.ipynb` and is not repeated here.

In [2]:
df = pd.read_csv("heart+disease/cleveland_clean.csv")
print("Loaded shape:", df.shape)
# Check that Arm C expects the same 297-record cleaned dataset used in Arm A and Arm B
df.head()

Loaded shape: (297, 15)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num,check
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0,False
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2,True
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1,True
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0,False
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0,False


## 4. Target construction

The original Cleveland target, `num`, represents the presence/severity of heart disease. For binary classification, observations with `num = 0` are coded as class 0 (absence of disease), while observations with `num > 0` are coded as class 1 (presence of disease). The derived `check` column is excluded from the feature matrix together with `num`, for the same leakage reason documented in Arm A.

In [3]:
target_col = "num"

y = (df[target_col] > 0).astype(int)
X = df.drop(columns=[target_col, "check"], errors="ignore")

print("Samples:", len(df), " Features:", X.shape[1])
print("Class balance:")
print(y.value_counts().rename({0: "no disease", 1: "disease"}))
print("Positive rate: {:.3f}".format(y.mean()))

Samples: 297  Features: 13
Class balance:
num
no disease    160
disease       137
Name: count, dtype: int64
Positive rate: 0.461


## 5. Feature definition

Two different groupings of the same 13 features are used in this notebook, for two different purposes:

- **Classifier feature grouping** — identical to Arm A and Arm B: continuous features are standardised, nominal category codes are one-hot encoded, and binary/count features pass through unchanged. This grouping is used inside every `Pipeline` that fits a logistic regression or random forest, exactly as in Arm A and Arm B, so the classifiers themselves see the same feature representation in all three arms.
- **Clustering feature grouping** — used only by k-prototypes, which requires features split into a numeric block (compared by Euclidean distance) and a categorical block (compared by simple matching), rather than one-hot encoded. `cp`, `restecg`, `slope`, and `thal` — the same four features one-hot encoded for the classifiers — are passed to k-prototypes as raw categorical codes: one-hot encoding them first would turn each category into several near-binary numeric columns under Euclidean distance, which is exactly what using k-prototypes over k-means was meant to avoid. `age`, `trestbps`, `chol`, `thalach`, and `oldpeak` are standardised and passed as numeric; `sex`, `fbs`, `exang`, and `ca` are passed as numeric unchanged, consistent with how they are treated as passthrough (not one-hot encoded) for the classifiers.

In [4]:
continuous  = ["age", "trestbps", "chol", "thalach", "oldpeak"]
nominal     = ["cp", "restecg", "slope", "thal"]
passthrough = ["sex", "fbs", "exang", "ca"]

# Clustering feature order: numeric block (continuous + passthrough) then
# categorical block (nominal). Categorical column indices are positional
# within this fixed order, not within the original dataframe.
cluster_numeric_cols     = continuous + passthrough
cluster_categorical_cols = nominal
cluster_feature_order    = cluster_numeric_cols + cluster_categorical_cols
cluster_categorical_idx  = list(range(len(cluster_numeric_cols), len(cluster_feature_order)))

print("Continuous:", continuous)
print("Nominal (one-hot for classifiers / categorical for clustering):", nominal)
print("Passthrough (already binary/count):", passthrough)
print("Clustering categorical column indices:", cluster_categorical_idx)

Continuous: ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
Nominal (one-hot for classifiers / categorical for clustering): ['cp', 'restecg', 'slope', 'thal']
Passthrough (already binary/count): ['sex', 'fbs', 'exang', 'ca']
Clustering categorical column indices: [9, 10, 11, 12]


## 6. Data-driven cluster exploratory summary (descriptive only — not used for evaluation)

Before nested cross-validation, k-prototypes is fitted once on the **entire** dataset purely to characterise what the algorithm finds, in the same spirit as Arm B's Section 6 sex-based summary. This whole-dataset fit is descriptive only: it gives an initial answer to RQ3 ("what did the data-driven method find, and does it recover the clinical variable used in Arm B?"), and it plays no role in the leakage-safe evaluation in Sections 10–14, where clustering is refit independently within each outer training fold. The candidate k is chosen the same way as in the per-fold procedure below: among the k's in `K_CLUSTER_CANDIDATES` for which every resulting cluster has at least `MIN_CLUSTER_SIZE` patients and at least `K_INNER` patients of each class, the one with the highest silhouette score is selected. The helper functions defined in this cell (`build_cluster_arrays`, `mixed_distance_matrix`, `fit_kprototypes_select_k`, `predict_clusters`) are reused unchanged by the per-fold procedure in Section 10.

In [5]:
def build_cluster_arrays(frame, scaler=None, fit_scaler=False):
    """Build the (numeric, categorical) arrays k-prototypes expects from a
    dataframe slice, in the fixed column order defined in Section 5. `scaler`
    is a StandardScaler for the five continuous columns only; sex/fbs/exang/ca
    are passed through unscaled, matching how they are treated as passthrough
    for the classifiers."""
    if fit_scaler:
        scaler = StandardScaler().fit(frame[continuous])
    cont_scaled = scaler.transform(frame[continuous])
    passthrough_vals = frame[passthrough].to_numpy(dtype=float)
    Xnum = np.hstack([cont_scaled, passthrough_vals])
    Xcat = frame[nominal].to_numpy()
    return Xnum, Xcat, scaler


def mixed_distance_matrix(Xnum_a, Xcat_a, Xnum_b, Xcat_b, gamma):
    """Pairwise dissimilarity between two point sets, using exactly the
    k-prototypes cost function (squared Euclidean on the numeric block plus
    gamma times simple-matching count on the categorical block), so the
    silhouette score used to select k is consistent with what k-prototypes
    itself minimises."""
    num_sq = ((Xnum_a[:, None, :] - Xnum_b[None, :, :]) ** 2).sum(axis=2)
    cat_mismatch = (Xcat_a[:, None, :] != Xcat_b[None, :, :]).sum(axis=2)
    return num_sq + gamma * cat_mismatch


def fit_kprototypes_select_k(frame, y_frame, k_candidates, min_cluster_size, random_state, n_init):
    """Fit k-prototypes on `frame`'s patients only, for each candidate k, and
    keep only the k's for which every resulting cluster has at least
    `min_cluster_size` patients AND at least K_INNER patients of each outcome
    class (the condition that actually makes 5-fold stratified inner CV
    well-defined within that cluster). Among the k's that pass, select the
    one with the highest silhouette score under the k-prototypes mixed
    distance. Returns (model, scaler, labels, k, silhouette), or
    (None, None, None, None, None) if no candidate k qualifies."""
    Xnum, Xcat, scaler = build_cluster_arrays(frame, fit_scaler=True)
    y_arr = np.asarray(y_frame)

    best = None
    for k in k_candidates:
        if len(frame) < min_cluster_size * k:
            continue  # cannot satisfy the size guard for this k regardless of split

        model = KPrototypes(n_clusters=k, init="Cao", n_init=n_init,
                             random_state=random_state, verbose=0)
        labels = model.fit_predict(np.hstack([Xnum, Xcat]), categorical=cluster_categorical_idx)

        sizes = np.bincount(labels, minlength=k)
        if sizes.min() < min_cluster_size:
            continue
        min_class_counts = [np.bincount(y_arr[labels == c], minlength=2).min() for c in range(k)]
        if min(min_class_counts) < K_INNER:
            continue

        dist = mixed_distance_matrix(Xnum, Xcat, Xnum, Xcat, model.gamma)
        sil = silhouette_score(dist, labels, metric="precomputed")

        if best is None or sil > best["silhouette"]:
            best = {"model": model, "scaler": scaler, "labels": labels, "k": k, "silhouette": sil}

    if best is None:
        return None, None, None, None, None
    return best["model"], best["scaler"], best["labels"], best["k"], best["silhouette"]


def predict_clusters(model, scaler, frame):
    """Assign patients in `frame` to the nearest already-fitted centroid.
    Never refits k-prototypes; used for validation-fold patients only."""
    Xnum, Xcat, _ = build_cluster_arrays(frame, scaler=scaler, fit_scaler=False)
    return model.predict(np.hstack([Xnum, Xcat]), categorical=cluster_categorical_idx)


# Descriptive only: fit once on all 297 patients to characterise what
# k-prototypes finds, independent of the leakage-safe per-fold procedure
# in Sections 10-14 below.
desc_model, desc_scaler, desc_labels, desc_k, desc_sil = fit_kprototypes_select_k(
    X, y, K_CLUSTER_CANDIDATES, MIN_CLUSTER_SIZE, RANDOM_STATE, N_INIT_KPROTO
)

if desc_model is None:
    print("No candidate k in", K_CLUSTER_CANDIDATES, "satisfied the size/class guards "
          "on the whole dataset; no descriptive clustering to report.")
else:
    print(f"Whole-dataset descriptive clustering selected k={desc_k} (silhouette={desc_sil:.3f})")

    desc = pd.DataFrame({
        "cluster": desc_labels,
        "sex": df["sex"].map({1.0: "male", 0.0: "female"}),
        "disease": y,
    })
    desc_summary = desc.groupby("cluster").agg(
        n=("disease", "size"),
        disease_rate=("disease", "mean"),
        pct_male=("sex", lambda s: (s == "male").mean()),
    ).round(3)
    desc_summary["pct_male"] = (desc_summary["pct_male"] * 100).round(1)
    print(desc_summary)
    desc_summary.to_csv("armC_cluster_descriptive_summary.csv")
    print("Saved armC_cluster_descriptive_summary.csv")

    print()
    print("Cluster membership vs sex:")
    desc_crosstab = pd.crosstab(desc_labels, desc["sex"], margins=True)
    print(desc_crosstab)
    desc_crosstab.to_csv("armC_cluster_vs_sex_crosstab.csv")
    print("Saved armC_cluster_vs_sex_crosstab.csv")

Whole-dataset descriptive clustering selected k=2 (silhouette=0.324)
           n  disease_rate  pct_male
cluster                             
0        159         0.226      66.7
1        138         0.732      68.8
Saved armC_cluster_descriptive_summary.csv

Cluster membership vs sex:
sex    female  male  All
row_0                   
0          53   106  159
1          43    95  138
All        96   201  297
Saved armC_cluster_vs_sex_crosstab.csv


## 7. Preprocessing

Preprocessing that estimates parameters from the data is kept inside a scikit-learn `Pipeline` together with each classifier, exactly as in Arm A and Arm B. Because a separate pipeline is fitted per outer fold per cluster (Section 11), the scaler and encoder only ever see the training portion of that cluster. The `ColumnTransformer` definition is identical to Arm A's and Arm B's — this is the classifier feature grouping from Section 5, not the clustering feature grouping.

In [6]:
# Fitted only inside the pipeline, on training-fold data (see Section 11), never on the full dataset.
preprocess = ColumnTransformer([
    ("num",  StandardScaler(),                       continuous),
    ("cat",  OneHotEncoder(handle_unknown="ignore"),  nominal),
    ("pass", "passthrough",                           passthrough),
])

## 8. Reuse Arm A's outer folds

The proposal requires identical outer cross-validation fold partitions across all arms (Section 3.5). Arm A already created and saved this partition to `fold_id.csv`; Arm C loads it directly rather than generating a new one, so every patient sits in exactly the same outer fold in Arm A, Arm B, and Arm C.

In [7]:
FOLD_FILE = "fold_id.csv"
if not os.path.exists(FOLD_FILE):
    raise FileNotFoundError(
        f"{FOLD_FILE} not found. Arm C requires the outer fold partition created by "
        "global_model_armA.ipynb; run that notebook first."
    )

fold_id = pd.read_csv(FOLD_FILE)["fold"].to_numpy()
if len(fold_id) != len(df):
    raise ValueError(
        f"{FOLD_FILE} has {len(fold_id)} entries but the current dataset has {len(df)} rows."
    )
print(f"Loaded outer fold assignment from {FOLD_FILE} (shared with Arm A and Arm B).")
print("Fold sizes:", np.bincount(fold_id))
print("Positives per fold:", np.bincount(fold_id[y.values == 1]))

Loaded outer fold assignment from fold_id.csv (shared with Arm A and Arm B).
Fold sizes: [60 60 59 59 59]
Positives per fold: [28 28 27 27 27]


## 9. Model definitions and hyperparameter grids

Exactly the same two classifiers and hyperparameter grids as Arm A and Arm B. Cluster models are tuned from the same candidate hyperparameters and the same selection procedure as the global model; only the data used to fit them, and how that data is grouped, differs.

In [8]:
models = {
    "logreg": (
        Pipeline([("pre", preprocess),
                  ("clf", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))]),
        {"clf__C": [0.01, 0.1, 1, 10]},
    ),
    "rf": (
        Pipeline([("pre", preprocess),
                  ("clf", RandomForestClassifier(random_state=RANDOM_STATE))]),
        {"clf__n_estimators": [200, 400], "clf__max_depth": [None, 5, 10]},
    ),
}

## 10. Per-fold clustering procedure and guards

Within each outer fold, clustering is fit using only that fold's training patients, with the same helper functions defined in Section 6 (`fit_kprototypes_select_k`, `predict_clusters`): the scaler for the continuous block and the k-prototypes model are both fit on training-fold patients only, and validation patients are assigned to the nearest already-fitted centroid via `.predict()` — never by refitting k-prototypes on data that includes them. This is the same leakage-prevention pattern Arm A and Arm B use for preprocessing, applied to an additional step: cluster formation itself.

Two guards, decided before running this notebook and applied identically to every fold:

- **Minimum cluster size and class count.** A candidate k is only accepted if every resulting training cluster has at least `MIN_CLUSTER_SIZE` (40) patients, *and* at least `K_INNER` (5) patients of each outcome class — the second condition is what actually makes 5-fold stratified inner cross-validation well-defined within that cluster, and is not implied by the first. Among the k's that pass both checks, the one with the highest silhouette score (computed under the same mixed-distance function k-prototypes itself minimises) is selected.
- **Fallback to the global model.** If no k in `{2, 3, 4}` passes both checks for a fold's training data, that fold falls back to treating its entire outer-training set as a single cluster — i.e. that fold's contribution to Arm C becomes identical in procedure to Arm A's global model for that fold. This is a pre-specified fallback, not a choice made after seeing which folds are difficult, and it is recorded per fold in Section 12 rather than applied silently.

A separate guard on the **validation** side handles single-class validation clusters: if the validation patients assigned to a cluster happen to contain only one outcome class, `roc_auc_score` is undefined for that cluster and is recorded as `NaN` in the cluster-specific breakdown, reusing the pattern Arm B applied to its sex-specific breakdown. This does not affect the primary pooled metric, which is computed once per fold across all clusters' validation patients combined.

In [9]:
def cluster_or_fallback(X_train_fold, y_train_fold):
    """Select cluster count and fit k-prototypes on this fold's outer-training
    patients only, applying the guards above. Returns
    (labels, scaler, model, k_used, fallback), where fallback=True means no
    candidate k satisfied the guards and the whole outer-training fold is
    treated as a single cluster (equivalent to the global model for this
    fold) — a rule fixed in advance, not chosen post hoc."""
    model, scaler, labels, k, sil = fit_kprototypes_select_k(
        X_train_fold, y_train_fold, K_CLUSTER_CANDIDATES, MIN_CLUSTER_SIZE,
        RANDOM_STATE, N_INIT_KPROTO
    )
    if model is None:
        return np.zeros(len(X_train_fold), dtype=int), None, None, 1, True
    return labels, scaler, model, k, False

## 11. Nested cross-validation within clusters

For each outer fold, clustering is fit once (Section 10) and reused for both classifiers, since cluster membership does not depend on which classifier will later be trained. Unlike Arm B, where the sex partition is fixed by definition and the original loop iterates model-then-fold, this notebook iterates fold-then-model so clustering is not refit identically twice per fold; this is a bookkeeping simplification specific to Arm C and does not change results.

Within each cluster, hyperparameters are selected using `GridSearchCV` with inner stratified 5-fold cross-validation on that cluster's outer-training patients only, exactly as in Arm A and Arm B; the outer validation fold is never seen during hyperparameter selection. ROC-AUC is the tuning criterion, matching Arm A and Arm B. Predicted class labels use the same fixed 0.5 probability threshold as the other two arms.

For every outer fold, every cluster's model predictions on that cluster's validation patients are pooled into a single population-level validation set before computing accuracy, F1, and ROC-AUC — the primary Arm C result, directly comparable to Arm A's and Arm B's pooled metrics. `best_params` is recorded per cluster per fold per model, following Arm B's practice, so that unstable or degenerate cluster models (analogous to Arm B's female-subgroup instability) can be diagnosed after the fact rather than only inferred from aggregate scores.

In [10]:
fold_results = []         # pooled, population-level: primary result
cluster_fold_results = [] # cluster-specific: secondary, descriptive result
cluster_choice = []       # which k (or fallback) each fold picked
best_params_rows = []     # per cluster per fold per model
oof_rows = []
patient_id = df.index.to_numpy()

for k in range(K_OUTER):
    train, validation = (fold_id != k), (fold_id == k)
    X_train_fold, y_train_fold = X[train], y[train]
    X_val_fold = X[validation]

    labels_train, scaler, cluster_model, k_used, fallback = cluster_or_fallback(X_train_fold, y_train_fold)

    if fallback:
        labels_val = np.zeros(len(X_val_fold), dtype=int)
        print(f"fold {k}: no candidate k in {K_CLUSTER_CANDIDATES} satisfied the "
              f"size/class guards (MIN_CLUSTER_SIZE={MIN_CLUSTER_SIZE}); "
              f"falling back to a single cluster (= global model) for this fold.")
    else:
        labels_val = predict_clusters(cluster_model, scaler, X_val_fold)
        print(f"fold {k}: selected k={k_used}, training cluster sizes "
              f"{np.bincount(labels_train, minlength=k_used).tolist()}")

    cluster_choice.append({
        "fold": k, "k_used": k_used, "fallback_to_global": fallback,
        "train_cluster_sizes": np.bincount(labels_train, minlength=k_used).tolist(),
        "val_cluster_sizes": np.bincount(labels_val, minlength=k_used).tolist(),
    })

    for name, (pipe, grid) in models.items():
        pooled_y_true, pooled_proba, pooled_pred = [], [], []

        for cluster_id in range(k_used):
            cluster_train = train.copy()
            cluster_train[train] = (labels_train == cluster_id)
            cluster_validation = validation.copy()
            cluster_validation[validation] = (labels_val == cluster_id)

            if cluster_validation.sum() == 0:
                continue  # no validation patients were assigned to this cluster this fold

            # Hyperparameter selection uses only this cluster's outer-training
            # patients; the outer validation fold is not seen until scoring below.
            inner = StratifiedKFold(n_splits=K_INNER, shuffle=True, random_state=RANDOM_STATE)
            search = GridSearchCV(pipe, grid, cv=inner, scoring="roc_auc", n_jobs=-1)
            search.fit(X[cluster_train], y[cluster_train])

            best = search.best_estimator_
            proba = best.predict_proba(X[cluster_validation])[:, 1]
            pred = (proba >= 0.5).astype(int)  # Fixed threshold, consistent with Arm A and Arm B.
            y_val = y[cluster_validation].to_numpy()

            cluster_fold_results.append({
                "model": name, "fold": k, "cluster": cluster_id,
                "n_train": int(cluster_train.sum()), "n_val": int(cluster_validation.sum()),
                "accuracy": accuracy_score(y_val, pred),
                "f1": f1_score(y_val, pred),
                "roc_auc": roc_auc_score(y_val, proba) if len(np.unique(y_val)) > 1 else np.nan,
            })
            best_params_rows.append({
                "model": name, "fold": k, "cluster": cluster_id,
                "n_train": int(cluster_train.sum()), "best_params": search.best_params_,
            })

            for pid, yt, p, c in zip(patient_id[cluster_validation], y_val, proba, pred):
                oof_rows.append({
                    "patient_id": int(pid), "fold": int(k), "cluster": int(cluster_id),
                    "y_true": int(yt), "model": name, "proba": float(p), "pred": int(c),
                })

            pooled_y_true.append(y_val)
            pooled_proba.append(proba)
            pooled_pred.append(pred)

        # Population-level pooled result for this outer fold: every cluster's
        # validation predictions are combined before scoring, not averaged,
        # so the pooled metric reflects the full validation fold at once.
        y_true_pooled = np.concatenate(pooled_y_true)
        proba_pooled = np.concatenate(pooled_proba)
        pred_pooled = np.concatenate(pooled_pred)

        fold_results.append({
            "model": name,
            "fold": k,
            "accuracy": accuracy_score(y_true_pooled, pred_pooled),
            "f1": f1_score(y_true_pooled, pred_pooled),
            "roc_auc": roc_auc_score(y_true_pooled, proba_pooled),
        })

fold_results_df = pd.DataFrame(fold_results)
cluster_fold_results_df = pd.DataFrame(cluster_fold_results)
cluster_choice_df = pd.DataFrame(cluster_choice)
best_params_df = pd.DataFrame(best_params_rows)
print()
print("Pooled nested cross-validation complete:", len(fold_results_df), "model x fold rows")
print("Cluster-specific nested cross-validation complete:", len(cluster_fold_results_df), "model x fold x cluster rows")

fold 0: selected k=2, training cluster sizes [109, 128]


fold 1: selected k=2, training cluster sizes [109, 128]


fold 2: selected k=2, training cluster sizes [115, 123]


fold 3: selected k=2, training cluster sizes [114, 124]


fold 4: selected k=2, training cluster sizes [110, 128]



Pooled nested cross-validation complete: 10 model x fold rows
Cluster-specific nested cross-validation complete: 20 model x fold x cluster rows


## 12. Fold-level results

The pooled, population-level fold results are the primary output of this notebook and are saved for comparison with Arm A and Arm B. The cluster-specific fold-level results are a secondary, descriptive breakdown, and the per-fold cluster choice (`k_used`, whether that fold fell back to the global model, and the resulting cluster sizes) is reported separately, since it is itself a result of applying a data-driven method to five different training folds rather than a fixed design choice.

In [11]:
fold_results_df.to_csv("armC_fold_results.csv", index=False)
print("Saved armC_fold_results.csv (pooled, primary)")
fold_results_df.round(3)

Saved armC_fold_results.csv (pooled, primary)


,model,fold,accuracy,f1,roc_auc
0,logreg,0,0.850,0.852,0.940
1,rf,0,0.883,0.881,0.934
2,logreg,1,0.883,0.873,0.907
3,rf,1,0.817,0.792,0.914
4,logreg,2,0.746,0.746,0.855
5,rf,2,0.746,0.727,0.853
6,logreg,3,0.780,0.745,0.872
7,rf,3,0.780,0.755,0.863
8,logreg,4,0.746,0.754,0.887
9,rf,4,0.797,0.760,0.920


In [12]:
cluster_choice_df.to_csv("armC_cluster_choice.csv", index=False)
print("Saved armC_cluster_choice.csv")
cluster_choice_df

Saved armC_cluster_choice.csv


,fold,k_used,fallback_to_global,train_cluster_sizes,val_cluster_sizes
0,0,2,False,"[109, 128]","[30, 30]"
1,1,2,False,"[109, 128]","[28, 32]"
2,2,2,False,"[115, 123]","[29, 30]"
3,3,2,False,"[114, 124]","[25, 34]"
4,4,2,False,"[110, 128]","[28, 31]"


In [13]:
cluster_fold_results_display = cluster_fold_results_df.round(3)
cluster_fold_results_display

,model,fold,cluster,n_train,n_val,accuracy,f1,roc_auc
0,logreg,0,0,109,30,0.867,0.926,0.752
1,logreg,0,1,128,30,0.833,0.286,0.877
2,rf,0,0,109,30,0.867,0.926,0.760
3,rf,0,1,128,30,0.900,0.400,0.802
4,logreg,1,0,109,28,0.821,0.872,0.862
5,logreg,1,1,128,32,0.938,0.875,0.911
6,rf,1,0,109,28,0.786,0.850,0.869
7,rf,1,1,128,32,0.844,0.615,0.880
8,logreg,2,0,115,29,0.862,0.913,0.875
9,logreg,2,1,123,30,0.633,0.154,0.667


In [14]:
best_params_df.to_csv("armC_best_params.csv", index=False)
print("Saved armC_best_params.csv")
best_params_df.head(10)

Saved armC_best_params.csv


,model,fold,cluster,n_train,best_params
0,logreg,0,0,109,{'clf__C': 0.1}
1,logreg,0,1,128,{'clf__C': 1}
2,rf,0,0,109,"{'clf__max_depth': 5, 'clf__n_estimators': 200}"
3,rf,0,1,128,"{'clf__max_depth': 10, 'clf__n_estimators': 200}"
4,logreg,1,0,109,{'clf__C': 0.1}
5,logreg,1,1,128,{'clf__C': 1}
6,rf,1,0,109,"{'clf__max_depth': None, 'clf__n_estimators': ..."
7,rf,1,1,128,"{'clf__max_depth': 5, 'clf__n_estimators': 200}"
8,logreg,2,0,115,{'clf__C': 0.1}
9,logreg,2,1,123,{'clf__C': 1}


## 13. Overall performance summary

Mean and standard deviation across the 5 outer folds are reported for each model, using the pooled population-level fold results, so that Arm C's primary comparison with Arm A and Arm B rests on the same kind of summary statistic in every notebook. The corresponding cluster-specific summary is reported separately below as a secondary, descriptive result.

In [15]:
results = (
    fold_results_df
    .groupby("model")[["accuracy", "f1", "roc_auc"]]
    .agg(["mean", "std"])
)
results.columns = ["accuracy_mean", "accuracy_std", "f1_mean", "f1_std", "rocauc_mean", "rocauc_std"]
results = results.reset_index()

for _, row in results.iterrows():
    print(f"{row['model']:7s} | ACC {row['accuracy_mean']:.3f} +/- {row['accuracy_std']:.3f}"
          f" | F1 {row['f1_mean']:.3f} +/- {row['f1_std']:.3f}"
          f" | AUC {row['rocauc_mean']:.3f} +/- {row['rocauc_std']:.3f}")

results.to_csv("armC_results.csv", index=False)
print("Saved armC_results.csv (pooled, primary)")
results.set_index("model").round(3)

logreg  | ACC 0.801 +/- 0.063 | F1 0.794 +/- 0.063 | AUC 0.892 +/- 0.033
rf      | ACC 0.804 +/- 0.051 | F1 0.783 +/- 0.060 | AUC 0.897 +/- 0.036
Saved armC_results.csv (pooled, primary)


,accuracy_mean,accuracy_std,f1_mean,f1_std,rocauc_mean,rocauc_std
model,,,,,,
logreg,0.801,0.063,0.794,0.063,0.892,0.033
rf,0.804,0.051,0.783,0.060,0.897,0.036


In [16]:
cluster_results = (
    cluster_fold_results_df
    .groupby(["cluster", "model"])[["accuracy", "f1", "roc_auc"]]
    .agg(["mean", "std"])
)
cluster_results.columns = ["accuracy_mean", "accuracy_std", "f1_mean", "f1_std", "rocauc_mean", "rocauc_std"]
cluster_results = cluster_results.reset_index()

cluster_results.to_csv("armC_cluster_specific_results.csv", index=False)
print("Saved armC_cluster_specific_results.csv (secondary, descriptive)")
print("Note: cluster ids are only comparable within the same fold, not across folds,")
print("since k-prototypes labels clusters independently each time it is fit; this table")
print("aggregates by cluster id anyway, as a rough per-slot summary, not by matched identity.")
cluster_results.round(3)

Saved armC_cluster_specific_results.csv (secondary, descriptive)
Note: cluster ids are only comparable within the same fold, not across folds,
since k-prototypes labels clusters independently each time it is fit; this table
aggregates by cluster id anyway, as a rough per-slot summary, not by matched identity.


,cluster,model,accuracy_mean,accuracy_std,f1_mean,f1_std,rocauc_mean,rocauc_std
0,0,logreg,0.806,0.076,0.879,0.046,0.866,0.070
1,0,rf,0.798,0.040,0.869,0.032,0.838,0.069
2,1,logreg,0.795,0.111,0.417,0.303,0.830,0.097
3,1,rf,0.809,0.073,0.392,0.154,0.835,0.064


## 14. Out-of-fold predictions

For every patient, the out-of-fold prediction from the one outer fold in which they were held out is retained, together with the patient identifier, fold id, cluster id (within that fold), true label, model name, predicted probability, and predicted class. This matches Arm A's and Arm B's out-of-fold prediction structure (`patient_id`, `fold`, `y_true`, `model`, `proba`, `pred`) with `cluster` added, so Arm A, Arm B, and Arm C can be compared using the same patient identifiers, fold assignment, and column layout. Because every validation patient is assigned to exactly one cluster in the fold that holds them out, each patient receives exactly one out-of-fold prediction per model.

In [17]:
oof_df = pd.DataFrame(oof_rows)

# Verify exactly one OOF prediction per patient per model, and full coverage of all patients.
counts_per_model = oof_df.groupby("model")["patient_id"].nunique()
assert (counts_per_model == len(df)).all(), "Every patient must receive exactly one OOF prediction per model."
assert oof_df.groupby(["model", "patient_id"]).size().max() == 1, "Duplicate OOF prediction detected for a patient."

oof_df.to_csv("armC_predictions.csv", index=False)
print("Saved armC_predictions.csv:", oof_df.shape)
oof_df.head()

Saved armC_predictions.csv: (594, 7)


,patient_id,fold,cluster,y_true,model,proba,pred
0,1,0,0,1,logreg,0.917070,1
1,2,0,0,1,logreg,0.916548,1
2,6,0,0,1,logreg,0.796407,1
3,8,0,0,1,logreg,0.786335,1
4,12,0,0,1,logreg,0.812578,1


## 15. Interpretation / notes

Every outer fold selected k=2 (`armC_cluster_choice.csv`), with the two training clusters close to evenly sized rather than skewed (e.g. 109/128, 115/123 patients). The whole-dataset descriptive clustering in Section 6 (`armC_cluster_descriptive_summary.csv`, `armC_cluster_vs_sex_crosstab.csv`) is retained for descriptive purposes only, characterising what k-prototypes finds; it plays no role in the leakage-safe evaluation reported in Sections 12-13. The cluster-specific breakdown (`armC_cluster_specific_results.csv`) shows one cluster with a markedly less stable F1 across folds than the other, worth keeping in mind when reading the pooled result.

For comparability: the same `fold_id.csv` partition, feature grouping, and classifiers/grids used here match Arm A and Arm B unchanged, so that `armC_results.csv` is directly comparable to `armA_results.csv` and `armB_results.csv` on the same terms, whenever that comparison is carried out.